# 07: El viaje de un dato: de su origen hasta una decisión de negocio

## [01] Detectar y eliminar duplicados

El pipeline se ejecutó dos veces por error. Deduplica quedándote con la PRIMERA aparición de cada id.

Resultado esperado

Original: 7 registros
Sin duplicados: 5 registros
Duplicados eliminados: 2

Resultado limpio:

| 101 | Mouse   | 25.00€  | lote 1 |
| --- | ------- | ------- | ------ |
| 102 | Teclado | 60.00€  | lote 1 |
| 103 | Monitor | 350.00€ | lote 1 |
| 104 | Webcam  | 80.00€  | lote 2 |
| 105 | Mouse   | 25.00€  | lote 2 |

In [26]:
# El pipeline se ejecutó dos veces: el lote 2 repite pedidos del lote 1.
pedidos = [
    {"id": 101, "producto": "Mouse",   "importe": 25.0,  "lote": 1},
    {"id": 102, "producto": "Teclado", "importe": 60.0,  "lote": 1},
    {"id": 101, "producto": "Mouse",   "importe": 25.0,  "lote": 2},
    {"id": 103, "producto": "Monitor", "importe": 350.0, "lote": 1},
    {"id": 102, "producto": "Teclado", "importe": 60.0,  "lote": 2},
    {"id": 104, "producto": "Webcam",  "importe": 80.0,  "lote": 2},
    {"id": 105, "producto": "Mouse",   "importe": 25.0,  "lote": 2},
]

In [27]:
vistos = set() # un conjunto: guarda cada id una sola vez
resultado = []

for pedido in pedidos:
    id = pedido['id']

    if id not in vistos:
        vistos.add(id)
        resultado.append(pedido)


In [28]:
# No toques
print(f"Original: {len(pedidos)} registros")
print(f"Sin duplicados: {len(resultado)} registros")
print(f"Duplicados eliminados: {len(pedidos) - len(resultado)}")
print("")
print("Resultado limpio:")
for r in resultado:
    print(f"  {r['id']} | {r['producto']:8} | {r['importe']:6.2f}€ | lote {r['lote']}")

Original: 7 registros
Sin duplicados: 5 registros
Duplicados eliminados: 2

Resultado limpio:
  101 | Mouse    |  25.00€ | lote 1
  102 | Teclado  |  60.00€ | lote 1
  103 | Monitor  | 350.00€ | lote 1
  104 | Webcam   |  80.00€ | lote 2
  105 | Mouse    |  25.00€ | lote 2


## [02] El proceso completo: reservas del restaurante

Reservas del día: filtra por fecha, valida nombre/personas, asigna mesa. La condición de rechazo está a medias: falta la otra mitad.

Resultado esperado

LISTA DE MESAS PARA HOY 

| 20:00 | García | 4 pers | Mesa mediana |
| ----- | ------ | ------ | ------------ |
| 21:15 | Ortega | 7 pers | Mesa grande  |
| 21:30 | López  | 2 pers | Mesa pequeña |

Reservas válidas: 3

Reservas rechazadas: 2

In [29]:
reservas_raw = [
    {"nombre": "García", "personas": 4,  "hora": "20:00", "fecha": "2026-01-15", "mesa": None},
    {"nombre": "López",  "personas": 2,  "hora": "21:30", "fecha": "2026-01-15", "mesa": None},
    {"nombre": "Martín", "personas": 6,  "hora": "20:30", "fecha": "2026-01-16", "mesa": None},
    {"nombre": "",       "personas": 3,  "hora": "22:00", "fecha": "2026-01-15", "mesa": None},
    {"nombre": "Ruiz",   "personas": -1, "hora": "21:00", "fecha": "2026-01-15", "mesa": None},
    {"nombre": "Ortega", "personas": 7,  "hora": "21:15", "fecha": "2026-01-15", "mesa": None},
]


In [30]:
hoy = "2026-01-15"

# EXTRACT: quédate solo con las reservas de hoy
reservas_hoy = [r for r in reservas_raw if r['fecha'] == hoy] # <- filtrar por r['fecha']

# TRANSFORM
reservas_validas = []
reservas_invalidas = []

for r in reservas_hoy:
    if not str(r['nombre']).strip() or r['personas'] <= 0:
        reservas_invalidas.append(r)
        continue

    if r['personas'] <= 2:
        r['mesa'] = "pequeña"
    elif r['personas'] <= 4:
        r['mesa'] = "mediana"
    else:
        r['mesa'] = "grande"

    reservas_validas.append(r)


In [31]:
# No toques: LOAD
print("=== LISTA DE MESAS PARA HOY ===")
for r in sorted(reservas_validas, key=lambda x: x["hora"]):
    print(f"  {r['hora']} | {r['nombre']:10} | {r['personas']} pers | Mesa {r['mesa']}")
print(f"\nReservas válidas: {len(reservas_validas)}")
print(f"Reservas rechazadas: {len(reservas_invalidas)}")

=== LISTA DE MESAS PARA HOY ===
  20:00 | García     | 4 pers | Mesa mediana
  21:15 | Ortega     | 7 pers | Mesa grande
  21:30 | López      | 2 pers | Mesa pequeña

Reservas válidas: 3
Reservas rechazadas: 2


## [03] Las tres zonas: raw → procesado → final

Sensores con errores (None y fuera de rango). Descarta los malos y calcula la media por sensor. La zona final se relee del fichero.

Resultado esperado

RAW: 6 registros guardados

PROCESADO: 4 válidos, 2 descartados

FINAL: promedio por sensor
- A1: 22.6°C
- B2: 23.2°C

In [64]:
import json, os
from collections import defaultdict

sensores_raw = [
    {"sensor": "A1", "temp": 22.5, "ts": "2026-01-15T10:00"},
    {"sensor": "B2", "temp": None, "ts": "2026-01-15T10:01"},
    {"sensor": "A1", "temp": 22.8, "ts": "2026-01-15T10:05"},
    {"sensor": "B2", "temp": 23.2, "ts": "2026-01-15T10:06"},
    {"sensor": "A1", "temp": 999.0, "ts": "2026-01-15T10:10"},
    {"sensor": "A1", "temp": 22.4, "ts": "2026-01-15T10:15"},
]


In [69]:
# ZONA RAW: guardar tal cual
data_raw_path = "data/raw/"

os.makedirs(data_raw_path, exist_ok=True)

with open(f'{data_raw_path}sensores.json', "w", encoding='utf-8') as f:
    json.dump(sensores_raw, f)
print(f"RAW: {len(sensores_raw)} registros guardados")


RAW: 6 registros guardados


In [84]:
# ZONA PROCESADA: descartar None y fuera de rango (-40 a 60)
data_processed_path = "data/processed/"

os.makedirs(data_processed_path, exist_ok=True)

datos_limpios = []
descartados = 0

for lectura in sensores_raw:
    temp = lectura['temp']

    if temp is None or not (-40 <= temp <= 60):
        descartados +=1
    else:
        datos_limpios.append(lectura) 

with open(f'{data_processed_path}sensore_limpios.json', "w", encoding='utf-8') as f:
    json.dump(datos_limpios, f)

print(f"PROCESADO: {len(datos_limpios)} válidos, {descartados} descartados")


PROCESADO: 4 válidos, 2 descartados


In [85]:
# ZONA FINAL: la media de cada sensor
data_final_path = "data/final/"

os.makedirs(f'{data_final_path}', exist_ok=True)

temperaturas = defaultdict(list)

for lectura in datos_limpios:
    temperaturas[lectura['sensor']].append(lectura['temp'])

resumen = {}

for s, temps in temperaturas.items():
    resumen[s] = round(sum(temps) / len(temps), 2)

with open(f'{data_final_path}resumen.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f)

In [86]:
# No toques: relee el fichero final para comprobar que se escribió
print("FINAL: promedio por sensor")
with open("data/final/resumen.json", "r", encoding="utf-8") as f:
    for sensor, promedio in json.load(f).items():
        print(f"  {sensor}: {promedio:.1f}°C")

FINAL: promedio por sensor
  A1: 22.6°C
  B2: 23.2°C


## [04] Que un dato malo no te tire el proceso entero

Registros con errores. La validación de nombre está hecha. Añade precio (float, > 0) y stock (int, >= 0). El pipeline sigue aunque un registro falle.

Resultado esperado

=== REPORTE DE CALIDAD ===
- Total recibidos: 6
- Procesados OK: 2 (33%)
- Con errores: 4 (67%)

Errores:
- PROD-002: could not convert string to float: 'invalido'
- PROD-003: nombre vacío
- PROD-004: stock negativo: -5
- PROD-006: precio inválido: 0.0


In [87]:
registros = [
    {"sku": "PROD-001", "nombre": "Laptop",  "precio": "1299.99", "stock": "50"},
    {"sku": "PROD-002", "nombre": "Mouse",   "precio": "invalido", "stock": "200"},
    {"sku": "PROD-003", "nombre": "",        "precio": "59.99",   "stock": "100"},
    {"sku": "PROD-004", "nombre": "Monitor", "precio": "449.00",  "stock": "-5"},
    {"sku": "PROD-005", "nombre": "Teclado", "precio": "89.90",   "stock": "75"},
    {"sku": "PROD-006", "nombre": "Cable",   "precio": "0",       "stock": "300"},
]

In [ ]:
procesados = []
errores = []

for reg in registros:
    try:
        # Tres validaciones
        if not reg['nombre'].strip():
            raise ValueError("Nombre vacío")

        precio =  float(reg['precio'])
        if precio <= 0:
            raise ValueError(f"precio inválido: {precio}")

        stock = int(reg['stock'])
        if stock < 0:
            raise ValueError(f"stock negativo: {stock}")

        procesados.append({"sku": reg['sku'], 
                           "nombre": reg["nombre"],
                           "precio": precio, 
                           "stock": stock
                           })
        
    except Exception as e:
        errores.append({"sku": reg['sku'], "error": str(e)})

errores

[{'sku': 'PROD-002', 'error': "could not convert string to float: 'invalido'"},
 {'sku': 'PROD-003', 'error': 'Nombre vacío'},
 {'sku': 'PROD-004', 'error': 'stock negativo: -5'},
 {'sku': 'PROD-006', 'error': 'precio inválido: 0.0'}]

In [94]:
# No toques
total = len(registros)
print("=== REPORTE DE CALIDAD ===")
print(f"Total recibidos: {total}")
print(f"Procesados OK: {len(procesados)} ({round(len(procesados)*100/total)}%)")
print(f"Con errores: {len(errores)} ({round(len(errores)*100/total)}%)")
print("")
print("Errores:")
for e in errores:
    print(f"  {e['sku']}: {e['error']}")

=== REPORTE DE CALIDAD ===
Total recibidos: 6
Procesados OK: 2 (33%)
Con errores: 4 (67%)

Errores:
  PROD-002: could not convert string to float: 'invalido'
  PROD-003: Nombre vacío
  PROD-004: stock negativo: -5
  PROD-006: precio inválido: 0.0
